**Lab 1 package (this notebook uses functions from it)**  
- **Package URL:** https://github.com/averybooks/radiolab2026-Cal (branch: `Nick-lab-1`)  
- **Installation command** (run the cell below once):

In [ ]:

!pip install "git+https://github.com/averybooks/radiolab2026-Cal.git@Nick-lab-1"

# Lab 1: Digital Sampling, Fourier Filtering, and Heterodyne Mixers in Radio Astronomy

**Name:** Nick Gibb
**Course:** Radio Astronomy Labratory
**Lab:** 1
**Date:** February 9th, 2026

## Overview
This notebook focuses on  digital sampling, Fourier transforms, spectral leakage, noise properties, and heterodyne mixers in the context of radio astronomy. Python is the language used for simulation, data analysis, and visualization of results in order to address the lab goals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from radiolab_utils import sample_sine, power_spectrum_simple, power_spectrum, load_run 


## 5.3. Digital Sampling and the Nyquist Criterion 

In order to gain a better understanding for digital sampling and aliasing, we start with a simulated sine wave. By varying the siganl frequency and the sampling rate, we are able to examine when the Nyquist Criterion is vioalted and how aliasing effects sampled data and its assosicated Fourier spectrum as well. 

For a signal sampled at frequency νₛ, the Nyquist Criterion ensures that the signal bandwidth is less than νₛ / 2. This is the limit in order to avoid aliasing. Any frequency above this limis will be relfected into lower frequencies, which then create strange spectral features.

We aim to understand how aliasing works in different contexts through the use of oscillatory simulations.

In [ ]:
f_sample = 1000
frequencies = [200, 400, 700]
plt.figure(figsize=(10, 6))

for i, f_sig in enumerate(frequencies):
    t, x = sample_sine(f_sig, f_sample)
    plt.subplot(3, 1, i + 1)
    plt.plot(t[:50], x[:50], marker='o')
    plt.title(f"Signal freuquency = {f_sig} Hz")
    plt.ylabel("Voltage")
plt.xlabel("Time(s)")
plt.tight_layout()
plt.show()

**Figure 1.** Time-domain samples of the simulated sine waves with frequencies ν₀ = 200 Hz, 400 Hz, and 700 Hz. All were sampled at all sampled at vₛ = 1000 Hz. Signals with frequencies below the Nyquist limit of 500 Hz are well resolved while the signal with a frequency of 700 Hz exhibits distorted oscillatory behavior. 

In [ ]:
def power_spectrum(x, f_sample):
    fft_vals = np.fft.fft(x)
    freqs = np.fft.fftfreq(len(x), 1/f_sample)
    return np.fft.fftshift(freqs), np.fft.fftshift(np.abs(fft_vals)**2)

plt.figure(figsize=(10,6))

for f_sig in frequencies:
    t, x = sample_sine(f_sig, f_sample)
    freqs, power = power_spectrum(x, f_sample)
    plt.plot(freqs, power, label = f"ν₀ = {f_sig} Hz")

plt.xlim(-600, 600)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.title("Aliasing in the Frequency Domain")
plt.legend()
plt.show()


**Figure 2.** The power spectrum of simulated sine waves sampled at νₛ = 1000 Hz. Signals with ν₀ = 200 Hz and singals with ν₀ = 400 Hz lie below the Nyquist frequency and are therefore shown with their true frequencies. However, the signal with ν₀ = 700 Hz exceeds the limit of νₛ / 2 and is aliased to ν_alias = |ν₀ − νₛ| = 300 Hz.

This demonstrates how frequency components above the Nyuquist limit are not simply taken away, but instead are reflected into lower Nyquist zones. This leads to ambiguity in determining the frequency if not given knowledge pertaining to the signal bandwidth. 

By increasing ν₀ beyond νₛ, additional Nyquist zones can be demonstrated, each folding the signal back into the baseband. With no filtering, signals from multiple Nyquist zones become impossible to tell apart, which motivates the use of analog anti-aliasing filters in actual recievers. 

## 5.3.1 Sampling with an SDR and Raspberry Pi

As now the behavior of ideal sampled signals has been shown, we move on to examining real data acquired with a software-defined radio (SDR). This SDR introduces analog filtering, finite dynamic range, and insturmental effects that modify the observed spectra. 

Through purposefully varying the sampling rate and signal frequency, we examine alaising behavior and show the SDR bandpass response. 

Data was collected using a Raspberry Pi running python to crease a ugradio.sdr.SDR object. Data was collected at varied sampling rates and saves to a Google Drive folder. The sampling rates included in our experiment were 1, 1.5, 2, 2.5, and 3 MHz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("Sin_wave_DATA/test_1.5MHz_300kHz.npz")
print(data.files)
print(data["arr_0"].shape)


In [ ]:
fs = 1.5e6  
x = data["arr_0"][0]  

t = np.arange(len(x)) / fs

plt.figure(figsize=(10,4))
plt.plot(t[:200], x[:200])
plt.xlabel("Time (s)")
plt.ylabel("Voltage (arb)")
plt.title("Time-domain SDR voltage")
plt.show()

**Figure 3.** Time-domain voltage samples aquired using the SDR for a sinusoidal input signal. The data are shown over a short time interval to emphasize the discrete sampling of the waveofrm at a high sampling rate. Differing from the ideal simulated sine waves, the SDR signal exhibits quantization effects and instrumental artifacts, which arise from the analog front-end and the internal filtering of the reciever.

In [ ]:
x = x - np.mean(x)  

fft_vals = np.fft.fft(x)
freqs = np.fft.fftfreq(len(x), d=1/fs)

power = np.abs(fft_vals)**2

freqs = np.fft.fftshift(freqs)
power = np.fft.fftshift(power)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(freqs/1e3, power)
plt.xlabel("Frequency (kHz)")
plt.ylabel("Power (arb)")
plt.title("Power spectrum: fs = 1.5 MHz, f0 = 300 kHz")
plt.yscale("log")
plt.xlim(-800, 800)
plt.show()

**Figure 4.** The power spectrum of a 300kHz sine wave sampled at fs = 1.5MHz using the SDR. The signal is well below the Nyquist frequency of 750kHz and is therefore recovered at its true frequency. The finite width of the spectral peak is caused by the limited number of samples and the absense of windowing, while the broadband backgorund reflects instrumental noise. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("Sin_wave_DATA/test_1MHz_500kHz.npz")

fs = 1.0e6
x = data["arr_0"][0]
x = x - np.mean(x)

fft_vals = np.fft.fft(x)
freqs = np.fft.fftfreq(len(x), d=1/fs)

power = np.abs(fft_vals)**2

freqs = np.fft.fftshift(freqs)
power = np.fft.fftshift(power)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(freqs/1e3, power)
plt.xlabel("Frequency (kHz)")
plt.ylabel("Power (arb)")
plt.title("Power spectrum: fs = 1.0 MHz, f0 = 500 kHz (Nyquist)")
plt.yscale("log")
plt.xlim(-600, 600)
plt.show()

**Figure 5.** The power spectrum of a 500 kHz sine wave sampled at fs = 1.0 MHz, which corresponds with the Nyquist frequency. The spectrum is very distorted and lacks any well defined spectral peak. This demonstrates that signals at or above the Nyquist limit cannot be reliably reconstructed. This behavior shows the breadown of sampling theory at the Nyquist boundary. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_run(npz_path, run_index=0):
    data = np.load(npz_path)
    x = data["arr_0"][run_index].astype(float)
    return x

def power_spectrum(x, fs, window=True):
    x = x - np.mean(x)  

    if window:
        w = np.hanning(len(x))
        x = x * w

    fft_vals = np.fft.fft(x)
    freqs = np.fft.fftfreq(len(x), d=1/fs)

    power = np.abs(fft_vals)**2

    freqs = np.fft.fftshift(freqs)
    power = np.fft.fftshift(power)
    return freqs, power

fs = 1.0e6  

runs = [
    ("test_1MHz_50kHz.npz",  "50 kHz"),
    ("test_1MHz_200kHz.npz", "200 kHz"),
    ("test_1MHz_350kHz.npz", "350 kHz"),
    ("test_1MHz_450kHz.npz", "450 kHz"),
    ("test_1MHz_500kHz.npz", "500 kHz (Nyquist)")
]

plt.figure(figsize=(10, 6))

for fname, label in runs:
    try:
        x = load_run("Sin_wave_DATA/" + fname, run_index=0)            
        freqs, pwr = power_spectrum(x, fs, window=True)

        plt.plot(freqs/1e3, pwr, label=label)
    except FileNotFoundError:
        print(f"Missing file: {fname} (skip)")

plt.axvline(fs/2/1e3, linestyle="--", linewidth=1)  
plt.axvline(-fs/2/1e3, linestyle="--", linewidth=1)  

plt.xlabel("Frequency (kHz)")
plt.ylabel("Power (arb)")
plt.title("Overlay Power Spectra (fs = 1.0 MHz): Approaching Nyquist")
plt.yscale("log")
plt.xlim(-600, 600)
plt.legend()
plt.tight_layout()
plt.show()

**Figure 6.** Overlay of power spectra for sine waves sampled at a fixed rate of fs ​= 1.0 MHz with signal frequencies from 50 kHZ to 500 kHz. As the signal frequency approaches the Nyquist limit at 500 kHZ (dashed lines), the spectral peaks broaden and distort due to folding of frequency components into the baseband. At the Nyquist frequency, the spectrum lacks a well-defined peak, demonstrating the breakdown of reliable frequency reconstruction at and past the Nyquist boundary. 

## 5.4 Voltage Spectra and Power Spectra

In this section, we examine the distinction between voltage spectra and power spectra and their respective roles in frequency-domain analysis. **Power spectra** summarize the distribution of signal energy as a function of frequency and can be integrated to quantify total power in a band. **Voltage spectra** preserve full phase information via the complex amplitude $X(f)$. The two are related by

$$
P(f) = |X(f)|^2 = X(f)\,X^*(f).
$$

We then explore the real and imaginary components of the voltage spectrum, the physical meaning of negative frequencies, and the Wiener–Khinchin theorem linking the power spectrum to the autocorrelation function. 

In practice, **power spectra** are preferred when the goal is to sum the strength of spectral features, compare signal and noise levels, or measure brightness temperature in radio astronomy. **Voltage spectra** are preferred when phase information is physically meaningful.

The complex voltage spectrum includes both the real and imaginary components. These represent the orthogonal phase components of the signal. The real part is tied to the in-phase component of the oscillation, and the imaginary part corresponds to the quadrature component, which is phase-shifted by 90 degrees.

The real and imaginary components are used for discovering the amplitude and instantaneous phase of each frequency component. The relative contributions of these components will then determine the phase angle of the signal in the frequency domain.


Negative frequencies come from expressing real valued signals as sums of complex exponentials. A real sinusoidal signal can be written as the superposition of two complex exponentials, producing symmetric frequency components at positive and negative frequencies.

For real valued time domain signals, the voltage spectrum satisfies the conjugate symmetry relation

$$
X(-f) = X^*(f),
$$

which helps to make it so that the inverse Fourier transform remains real. As a result, negative frequencies do not represent additional physical oscillations, but rather encode phase information necessary for a complex representation of the signal.


#### Real and Imaginary components of the Voltage Spectrum

To examine the structure of the complex voltage spectrum, we plot the real and imaginary components separately.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("Sin_wave_DATA/test_1.5MHz_300kHz.npz")
fs = 1.5e6
x = data["arr_0"][0]
x = x - np.mean(x)

spec = np.fft.fft(x)
freqs = np.fft.fftfreq(len(x), d=1/fs)

spec = np.fft.fftshift(spec)
freqs = np.fft.fftshift(freqs)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(freqs/1e3, spec.real, label="Real part")
plt.plot(freqs/1e3, spec.imag, label="Imaginary part")

plt.xlabel("Frequency (kHz)")
plt.ylabel("Voltage Spectrum (arb)")
plt.title("Real and Imaginary Components of the Voltage Spectrum")
plt.legend()
plt.xlim(-800, 800)
plt.show()

**Figure 7.** Real (in-phase) and imaginary (quadrature) components of the complex voltage spectrum for a 300 kHz sine wave sampled at 1.5 MHz using the SDR. The real component is an even function of frequency, while the imaginary component is an odd function. This shows the conjugate symmetry that is required for a real-valued time-domain signal.

The real and imaginary components of the voltage spectrum clearly exhibit the symmetry of the positive and negative frequencies. The real component is symmetric about zero frequency, showcasing an even function. The imaginary component is antisymmetric, proving to be an odd function. This conjugate symmetry ensures that the inverse transform remains real valued. The variations between the two components reflect the phase of the signal at each frequency. The mixture above demonstrates a phase offset introduced by the sampling process and instrument response. 

#### Power spectrum construction

The power spectrum is computed as $P(f) = |X(f)|^2$ from the unshifted FFT so that the same ordering is used for the inverse transform. Taking the squared magnitude throws away all phase information, and only the energy in each frequency bin is kept. In radio astronomy, power spectra can be used to measure flux density and brightness temperature as a function of frequency.

In [ ]:
spec_unshifted = np.fft.fft(x)
power = np.abs(spec_unshifted)**2
freqs_plot = np.fft.fftshift(np.fft.fftfreq(len(x), d=1/fs))
power_plot = np.fft.fftshift(power)

plt.figure(figsize=(10, 5))
plt.semilogy(freqs_plot/1e3, power_plot + 1e-30)
plt.xlabel("Frequency (kHz)")
plt.ylabel("Power (arb, log scale)")
plt.title("Power Spectrum (unshifted FFT, displayed shifted)")
plt.xlim(-800, 800)
plt.grid(True, alpha=0.3)
plt.show()

**Figure 8.** Power spectrum $P(f) = |X(f)|^2$ for the same SDR time series (300 kHz, $f_s = 1.5$ MHz), on a logarithmic scale. The peak at ±300 kHz contains most of the signal power. Phase information is lost, and only the magnitude at each frequency is retained. The broadband floor is instrumental and environmental noise.

#### Inverse FFT of power spectrum (Wiener–Khinchin)

The **correlation theorem** states that the Fourier transform of the autocorrelation function equals the power spectrum: $\mathcal{F}[\mathrm{ACF}(\tau)] = P(f)$. Therefore the inverse Fourier transform of $P(f)$ yields the ACF. We use the **unshifted** power spectrum so that `numpy.fft.ifft` receives the same frequency ordering as `numpy.fft.fft` outputs. The result is **real valued** because $P(f)$ is real and even in frequency for a real time series. This means that the IFT of $P$ has no imaginary component.

In [ ]:
spec_unshifted = np.fft.fft(x)
power = np.abs(spec_unshifted)**2



In [ ]:
acf_fft = np.fft.ifft(power)
acf_fft = np.real(acf_fft)  

In [ ]:
lags = np.fft.fftshift(np.fft.fftfreq(len(x), d=fs/len(x)))
acf_shifted = np.fft.fftshift(acf_fft)

plt.figure(figsize=(10, 5))
plt.plot(lags*1e6, acf_shifted.real)
plt.xlabel("Lag (μs)")
plt.ylabel("ACF (arb)")
plt.title("Inverse FFT of Power Spectrum (ACF via Wiener–Khinchin)")
plt.xlim(-50, 50)
plt.grid(True, alpha=0.3)
plt.show()

**Figure 9.** Autocorrelation function obtained by inverse Fourier transform of the power spectrum (Wiener–Khinchin theorem). The ACF is real and symmetric in lag. The peak at zero lag is the total power. The oscillatory sidelobes at nonzero lags show the sinusoidal content of the signal.

#### Autocorrelation 

The autocorrelation function can also be computed directly from the time series. We use `numpy.correlate(x, x, mode='full')` and `scipy.signal.correlate(x, x, mode='full')` to compute the linear correlation at all lags. The result has length $2N-1$ with zero lag at the center.

In [ ]:
acf_numpy = np.correlate(x, x, mode="full")
acf_scipy = signal.correlate(x, x, mode="full")
lags_full = (np.arange(2*n - 1) - (n - 1)) / fs

plt.figure(figsize=(10, 5))
plt.plot(lags_full*1e6, acf_numpy, label="numpy.correlate", alpha=0.8)
plt.plot(lags_full*1e6, acf_scipy, label="scipy.signal.correlate", linestyle="--", alpha=0.8)
plt.xlabel("Lag (μs)")
plt.ylabel("ACF (arb)")
plt.title("Direct Autocorrelation (linear, full mode)")
plt.legend()
plt.xlim(-50, 50)
plt.grid(True, alpha=0.3)
plt.show()

#### Correlation theorem verification

Overlaying the FFT-based ACF (inverse of power spectrum) with the direct correlation tests the correlation theorem. The **FFT-based ACF** is the **circular** (periodic) autocorrelation: the DFT assumes the signal repeats with period $N$, so lags wrap around. The **direct** `correlate(..., mode='full')` computes the **linear** ACF with no wrap-around. They agree well near zero lag. At large lags the circular ACF is 'poisoned' by wrapped contributions and can diverge from the linear ACF.

In [ ]:
acf_fft_norm = acf_shifted / np.max(np.abs(acf_shifted))
acf_direct_norm = acf_numpy / np.max(np.abs(acf_numpy))

plt.figure(figsize=(10, 5))
plt.plot(lags*1e6, acf_fft_norm.real, label="ACF from IFT of P(f) (circular)", linewidth=2)
plt.plot(lags_full*1e6, acf_direct_norm, label="Direct ACF (linear)", alpha=0.8)
plt.xlabel("Lag (μs)")
plt.ylabel("ACF (normalized)")
plt.title("Correlation Theorem: FFT-based vs Direct ACF")
plt.legend()

plt.xlim(-200, 200)
plt.grid(True, alpha=0.3)
plt.show()

**Figure 10.** FFT-based ACF (circular) vs direct linear ACF, normalized. Near zero lag they agree. However, at larger lags the circular ACF deviates because the DFT treats the record as one period of a periodic signal, so contributions from the “other end” of the record wrap into the ACF. Zero-padding the time series before the FFT reduces this wrap-around and improves agreement.

#### Zero-padding experiment

Padding the time series with zeros before computing the FFT increases the length of the transform. The power spectrum and its IFT then have more points. Rhe circular ACF effectively has a longer period, so wrap around is pushed to larger lags. As a result, the FFT-based ACF agrees better with the linear (direct) ACF over a wider range of lags.

In [ ]:
N_pad = 2 * len(x)
x_pad = np.zeros(N_pad)
x_pad[:len(x)] = x - np.mean(x) 

power_pad = np.abs(np.fft.fft(x_pad))**2
acf_pad = np.fft.ifft(power_pad).real
lags_pad = np.fft.fftshift(np.fft.fftfreq(N_pad, d=fs/N_pad))
acf_pad_shifted = np.fft.fftshift(acf_pad)

acf_pad_norm = acf_pad_shifted / np.max(np.abs(acf_pad_shifted))

plt.figure(figsize=(10, 5))
plt.plot(lags_pad*1e6, acf_pad_norm, label="ACF from IFT of P(f), zero-padded (circular)")
plt.plot(lags_full*1e6, acf_direct_norm, label="Direct ACF (linear)", alpha=0.8)
plt.xlabel("Lag (μs)")
plt.ylabel("ACF (normalized)")
plt.title("Zero-Padding: Improved Agreement Between FFT-based and Direct ACF")
plt.legend()

plt.xlim(-200, 200)
plt.grid(True, alpha=0.3)
plt.show()

**Figure 11.** With zero-padding (length $2N$), the FFT-based ACF matches the direct linear ACF over a wider range of lags. The circular period is effectively doubled, so wrap around artifacts appear at larger lags and the central region agrees better with the true linear ACF. In practice, padding is a simple way to make the correlation-theorem ACF more comparable to the direct correlation when only a finite record is available.

## 5.5 Leakage Power

The DFT (and FFT) computes the spectrum at $N$ frequencies for $N$ samples, with a fixed frequency separation $\Delta\nu = \nu_s/N$. When the signal frequency $\nu_0$ does **not** fall exactly on a frequency bin, power appears at other frequencies. This is known as **spectral leakage**. This section demonstrates leakage using a sine wave whose frequency lies between FFT bins, and explains it using the Convolution Theorem.

We use a short record of a pure sine wave at a frequency that is **not** an integer multiple of $\Delta\nu$. The FFT then samples the spectrum at discrete bins. The true spectrum of a finite length sinusoid is a sinc like pattern centered at $\nu_0$, so power is spread into adjacent and distant bins when $\nu_0$ is off-grid.

In [ ]:
fs_demo = 1000
N_demo = 128
df = fs_demo / N_demo  
f_on = 50.0
f_off = 53.0
t_demo = np.arange(N_demo) / fs_demo
x_on = np.sin(2 * np.pi * f_on * t_demo)
x_off = np.sin(2 * np.pi * f_off * t_demo)

fft_on = np.fft.fft(x_on)
fft_off = np.fft.fft(x_off)
freqs_demo = np.fft.fftshift(np.fft.fftfreq(N_demo, d=1/fs_demo))
P_on = np.fft.fftshift(np.abs(fft_on)**2)
P_off = np.fft.fftshift(np.abs(fft_off)**2)

plt.figure(figsize=(12, 5))
plt.semilogy(freqs_demo, P_on + 1e-30, label=r"$\nu_0 = 50$ Hz (on bin)")
plt.semilogy(freqs_demo, P_off + 1e-30, label=r"$\nu_0 = 53$ Hz (off bin)")
plt.axvline(f_on, color="gray", linestyle="--", alpha=0.7)
plt.axvline(f_off, color="gray", linestyle="--", alpha=0.7)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power (arb, log scale)")
plt.title("Spectral Leakage: Power at Frequencies Other Than ν₀ When Off Grid")
plt.legend()
plt.xlim(-100, 100)
plt.grid(True, alpha=0.3)
plt.show()

**Figure 12.** Power spectra (log scale) for two sinusoids: one at 50 Hz (aligned with an FFT bin) and one at 53 Hz (between bins). For the on-grid case, almost all power lies in one bin. For the off-grid case, power **leaks** into many bins (sidelobes). Power at $\nu \neq \nu_0$ is spectral leakage.

#### Convolution Theorem and spectral leakage

Truncating the time series to $N$ samples is equivalent to multiplying the true signal by a **rectangular window** of length $T = N/\nu_s$. The Fourier transform of that window is a **sinc** function. By the **Convolution Theorem**, multiplication in time is convolution in frequency, meaning the spectrum of the truncated signal is the convolution of the true spectrum (a delta at $\nu_0$) with the sinc. So the single line at $\nu_0$ is **smeared** into a sinc pattern. The DFT then **samples** that sinc at the discrete frequencies $k\Delta\nu$. When $\nu_0$ is not one of those samples, we see power at many bins. That is the mathematical origin of spectral leakage.

## 5.6 Frequency Resolution

The **frequency resolution** of a DFT is the minimum separation between two spectral lines that can be distinguished. For an FFT with $N$ samples and sampling rate $\nu_s$, the frequency spacing is $\Delta\nu = \nu_s/N$. Two sinusoids are resolvable when their frequency separation is similar to or larger than $\Delta\nu$. We use **lab two-tone data**: two signal generators at 300 kHz and at an offset (275, 295, 305, 345, or 385 kHz) combined and sampled with the SDR. The Convolution Theorem explains resolution. A longer window gives a narrower sinc in frequency.

In [ ]:
fs_res = 1.5e6
data_275 = np.load("Sin_freq_delta_DATA/sinoffset300_275.npz")
data_305 = np.load("Sin_freq_delta_DATA/sinoffset300_305.npz")
x_275 = data_275["arr_0"][0].astype(float) - np.mean(data_275["arr_0"][0])
x_305 = data_305["arr_0"][0].astype(float) - np.mean(data_305["arr_0"][0])
N_res = len(x_275)
freqs_res = np.fft.fftshift(np.fft.fftfreq(N_res, d=1/fs_res))
P_275 = np.fft.fftshift(np.abs(np.fft.fft(x_275))**2)
P_305 = np.fft.fftshift(np.abs(np.fft.fft(x_305))**2)
df_res = fs_res / N_res

plt.figure(figsize=(10, 5))
plt.semilogy(freqs_res/1e3, P_275 + 1e-30, label=r"Tones at 300 kHz & 275 kHz ($\Delta\nu$ = 25 kHz)")
plt.semilogy(freqs_res/1e3, P_305 + 1e-30, label=r"Tones at 300 kHz & 305 kHz ($\Delta\nu$ = 5 kHz)")
plt.axvline(275, color="gray", linestyle="--", alpha=0.5)
plt.axvline(300, color="gray", linestyle="--", alpha=0.5)
plt.axvline(305, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("Frequency (kHz)")
plt.ylabel("Power (arb, log)")
plt.title(f"Frequency Resolution: Two-Tone SDR Data (Δν = fs/N ≈ {df_res/1e3:.1f} kHz)")
plt.legend()
plt.xlim(200, 400)
plt.grid(True, alpha=0.3)
plt.show()

**Figure 13.** Power spectrum of lab two-tone data: 300 kHz with 275 kHz (25 kHz spacing) and 300 kHz with 305 kHz (5 kHz spacing). Frequency resolution is $\Delta\nu = \nu_s/N \approx 732$ Hz per bin. The two peaks are clearly resolved when the tone separation is larger than $\Delta\nu$. For very close tones, main lobes overlap. The minimum distinguishable spacing is of order $1/T$ where $T$ is the observation time.

## 5.7 Power Spectra in Other Nyquist Windows

The standard FFT returns the spectrum only in the first Nyquist zone, $-\nu_s/2$ to $+\nu_s/2$. The **bandwidth** of the signal, which is including both positive and negative frequencies, must not exceed $\nu_s$. The **center** frequency can lie in a higher Nyquist zone. Computing the spectrum at frequencies in other zones (e.g. $\pm W\nu_s/2$ with $W \geq 4$) requires evaluating the DFT at those frequencies.For example, with `ugradio.dft` or a manual sum—rather than the fixed FFT grid. The same time series can then be interpreted in different Nyquist windows. Lab 4 uses a higher window (e.g. 12th) because the band of interest is centered there while the bandwidth still fits within $\nu_s$. If the bandwidth exceeds $\nu_s$, aliasing folds power from other zones into the baseband and the spectrum is no understandable.

## 5.8 Fourier Transforms of Noise

In radio astronomy, **noise** usually means random fluctuations from a known statistical distribution (e.g. Gaussian from Johnson noise). We analyze the properties of such noise, its distribution in the time domain, the power spectrum, and the ACF. Below we use **lab noise data** from the laboratory noise generator, band-pass filtered to prevent aliasing before direct sampling.

In [ ]:
noise_data = np.load("Noise_DATA/noise_bpf.npz")
noise_blocks = noise_data["arr_0"] 
noise = noise_blocks.flatten()
noise = noise.astype(float) - np.mean(noise)
fs_noise = 1.5e6
N_noise = len(noise)
print(f"Mean: {np.mean(noise):.6f}, RMS: {np.std(noise):.4f}, N = {N_noise}")

counts, bins = np.histogram(noise, bins=50, density=True)
x_gauss = np.linspace(noise.min(), noise.max(), 200)
sig = np.std(noise)
gauss_pdf = (1 / (sig * np.sqrt(2*np.pi))) * np.exp(-0.5 * (x_gauss/sig)**2)
plt.figure(figsize=(8, 4))
plt.hist(noise, bins=50, density=True, alpha=0.7, label="Noise histogram (lab)")
plt.plot(x_gauss, gauss_pdf, "r-", label="Gaussian (σ = RMS)")
plt.xlabel("Voltage (arb)")
plt.ylabel("Density")
plt.title("Noise Amplitude Distribution (Lab Noise Generator + BPF)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Figure 14.** Histogram of lab noise (band-pass filtered). The overplotted curve is a Gaussian with width equal to the sample RMS. Receiver noise is often approximately Gaussian due to the Central Limit Theorem. Deviations from the curve can indicate clipping, interference, or non-Gaussian sources.

In [ ]:
block_len = noise_blocks.shape[1]
n_blocks = noise_blocks.shape[0]
P_blocks = []
for k in range(n_blocks):
    blk = noise_blocks[k].astype(float) - np.mean(noise_blocks[k])
    P_blocks.append(np.abs(np.fft.fft(blk))**2)
P_avg = np.mean(P_blocks, axis=0)
freqs_noise = np.fft.fftshift(np.fft.fftfreq(block_len, d=1/fs_noise))
P_single = np.fft.fftshift(P_blocks[0])
P_avg_shifted = np.fft.fftshift(P_avg)

plt.figure(figsize=(10, 5))
plt.semilogy(freqs_noise/1e3, P_single + 1e-30, alpha=0.7, label="Single block")
plt.semilogy(freqs_noise/1e3, P_avg_shifted + 1e-30, label=f"Average of {n_blocks} blocks")
plt.xlabel("Frequency (kHz)")
plt.ylabel("Power (arb, log)")
plt.title("Noise Power Spectrum: Single Block vs Time Average")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Figure 15.** Power spectrum of noise for a single block and for the average of 16 blocks. Averaging reduces the variance of the spectrum,resulting in a smoother curve. The mean power level is unchanged. For a weak signal in noise, time averaging improves the effective SNR (radiometer equation: SNR $\propto \sqrt{\Delta\nu\,\tau}$ for integration time $\tau$ and bandwidth $\Delta\nu$).

In [ ]:
blk = noise_blocks[0].astype(float) - np.mean(noise_blocks[0])
acf_noise = np.correlate(blk, blk, mode="full")
acf_noise = acf_noise / np.max(acf_noise)
lags_noise = (np.arange(len(acf_noise)) - (block_len-1)) / fs_noise

plt.figure(figsize=(10, 4))
plt.plot(lags_noise*1e6, acf_noise)
plt.xlabel("Lag (μs)")
plt.ylabel("ACF (normalized)")
plt.title("Autocorrelation of Noise Block")
plt.xlim(-20, 20)
plt.grid(True, alpha=0.3)
plt.show()

Autocorrelation of a single block of (zero-mean) noise. The ACF is normalized to a maximum of 1 at zero lag. For uncorrelated white noise, the ACF is sharply peaked at zero lag and falls to near zero at nonzero lags. The small fluctuations at nonzero lags are due to finite sample size. This contrasts with a sinusoidal signal, whose ACF has a central peak plus oscillatory sidelobes. The width of the central peak is related to the bandwidth of the noise (or the coherence time of the process). The ACF and the power spectrum are Fourier transform pairs (Wiener–Khinchin), so the FWHM of the ACF central peak ($\Delta\tau_{\mathrm{FWHM}}$) and the FWHM of the power spectrum ($\Delta f_{\mathrm{FWHM}}$) are related: $\Delta f_{\mathrm{FWHM}} \cdot \Delta\tau_{\mathrm{FWHM}} \sim \mathrm{constant}$ (uncertainty relation). A narrower ACF peak therefore corresponds to a broader spectrum, and vice versa.

---

## 6. Fourier Transforms: Analytic and Discrete (Theory)

This section summarizes the theoretical framework used in the lab. The **analytic** Fourier transform operates on continuous functions, and the **discrete** Fourier transform (DFT) and its fast implementation (FFT) operate on sampled data and allow for nearly all of our spectral analysis.

### 6.1 The Analytic Fourier Transform

The Fourier transform maps a time-domain signal $E(t)$ to the frequency domain:
$$
\tilde{E}(\nu) = \int_{-T/2}^{T/2} E(t)\, e^{-2\pi i\nu t}\, dt.
$$
Because of the imaginary exponent, $\tilde{E}(\nu)$ is complex-valued. The transform is **invertible**:
$$
E(t) = \frac{1}{B} \int_{-B/2}^{B/2} \tilde{E}(\nu)\, e^{2\pi i\nu t}\, d\nu.
$$
Key properties: **linearity** (FT of a sum is the sum of FTs), **invertibility** (no information lost), and **unitarity** (power preserved). In practice, finite $T$ and $B$ lead to spectral leakage and limits on resolution.

### 6.2 The Discrete Fourier Transform (DFT)

For $N$ samples at spacing $\Delta t = 1/\nu_s$, the integral becomes a sum. The DFT gives $N$ frequency bins with spacing $\Delta\nu = \nu_s/N$. In Python, `numpy.fft.fft` uses the standard FFT ordering (0 to $\nu_s/2$, then negative frequencies); `numpy.fft.fftshift` reorders for plotting. The `ugradio.dft` module allows **arbitrary** frequency grids, which are slower but flexible, for oversampling or evaluating spectra in other Nyquist windows.

### 6.3 Power Spectra and the DFT

The power spectrum is $P_\nu = \tilde{E}(\nu)\tilde{E}^*(\nu) = |\tilde{E}(\nu)|^2$. In code: `P = np.abs(fft(x))**2` or `P = (fft(x) * fft(x).conj()).real`. Power is real and non-negative; phase is discarded. Integrating $P$ over frequency gives total power (Parseval).

### 6.4 The Power Spectrum and the Autocorrelation Function (ACF)

The **correlation theorem** states that the Fourier transform of the autocorrelation function equals the power spectrum: $\mathcal{F}[\mathrm{ACF}] = P(\nu)$. Thus the ACF is the inverse FT of $P$. For finite data, the DFT yields a **circular** ACF. Zero-padding reduces wrap-around so it better matches the **linear** ACF from direct correlation (as demonstrated in §5.4).

## 7. Mixers (Lab Activity, Week 2)

A **mixer** multiplies two input voltages: the **RF** (signal) and the **LO** (local oscillator). The output contains **sum** and **difference** frequencies, $\nu_\mathrm{RF} \pm \nu_\mathrm{LO}$. We use lab data from a double-sideband (DSB) mixer and a single-sideband (SSB) mixer to demonstrate these concepts.

### 7.1 The Double-Sideband (DSB) Mixer

In our setup, $\nu_\mathrm{LO} = 300$ kHz and $\nu_\mathrm{RF} = \nu_\mathrm{LO} \pm \Delta\nu$ with $\Delta\nu$ small (e.g. 15 kHz). The mixer output is sampled and we compute the power spectrum. We expect peaks at the **difference** frequency $|\nu_\mathrm{RF} - \nu_\mathrm{LO}| = \Delta\nu$ and at the **sum** $\nu_\mathrm{RF} + \nu_\mathrm{LO}$. 

In [ ]:
fs_dsb = 2.4e6
dsb_lsb = np.load("DSB_DATA/fir_300_285_0.npz")["arr_0"]
dsb_usb = np.load("DSB_DATA/fir_300_315_0.npz")["arr_0"]
x_lsb = dsb_lsb[0].astype(float) - np.mean(dsb_lsb[0])
x_usb = dsb_usb[0].astype(float) - np.mean(dsb_usb[0])
N_dsb = len(x_lsb)
freqs_dsb = np.fft.fftshift(np.fft.fftfreq(N_dsb, d=1/fs_dsb))
P_lsb = np.fft.fftshift(np.abs(np.fft.fft(x_lsb))**2)
P_usb = np.fft.fftshift(np.abs(np.fft.fft(x_usb))**2)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(freqs_dsb/1e3, P_lsb)
axes[0].set_xlim(-100, 100)
axes[0].set_ylabel("Power (arb)")
axes[0].set_title(r"DSB Mixer: $\nu_\mathrm{RF} = 285$ kHz (Lower Sideband, $\Delta\nu = -15$ kHz)")
axes[0].axvline(15, color="gray", linestyle="--", alpha=0.7, label="Difference 15 kHz")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(freqs_dsb/1e3, P_usb)
axes[1].set_xlim(-100, 100)
axes[1].set_xlabel("Frequency (kHz)")
axes[1].set_ylabel("Power (arb)")
axes[1].set_title(r"DSB Mixer: $\nu_\mathrm{RF} = 315$ kHz (Upper Sideband, $\Delta\nu = +15$ kHz)")
axes[1].axvline(-15, color="gray", linestyle="--", alpha=0.7)
axes[1].axvline(15, color="gray", linestyle="--", alpha=0.7, label="Difference ±15 kHz")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Figure 16.** Power spectra of the DSB mixer output for lower sideband (RF = 285 kHz) and upper sideband (RF = 315 kHz). The **difference** frequency at ±15 kHz appears in both. The DSB mixer does not distinguish upper vs lower sideband, as both map to the same IF. The **sum** frequency (585 kHz, 615 kHz) lies outside this zoom. A low-pass or band-pass filter at the mixer output typically selects only the difference for further processing.

In [ ]:
t_dsb = np.arange(N_dsb) / fs_dsb
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(t_dsb[:200]*1e6, x_usb[:200])
axes[0].set_xlabel("Time (μs)")
axes[0].set_ylabel("Voltage (arb)")
axes[0].set_title("DSB Mixer Output (Upper Sideband): Time Series")
axes[0].grid(True, alpha=0.3)

fft_usb = np.fft.fft(x_usb)
freqs_fft = np.fft.fftfreq(N_dsb, d=1/fs_dsb)
sum_band = (np.abs(freqs_fft) >= 500e3) & (np.abs(freqs_fft) <= 700e3)
fft_filtered = fft_usb.copy()
fft_filtered[sum_band] = 0
x_filtered = np.fft.ifft(fft_filtered).real
axes[1].plot(t_dsb[:200]*1e6, x_filtered[:200])
axes[1].set_xlabel("Time (μs)")
axes[1].set_ylabel("Voltage (arb)")
axes[1].set_title("After Fourier Filtering (Sum Frequency Zeroed)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Figure 17.** Top: time-domain waveform of the DSB mixer output (upper sideband). Bottom: after zeroing the sum-frequency components in the Fourier domain and inverse transforming, the filtered signal retains mainly the **difference** (beat) frequency at 15 kHz. This illustrates how a low-pass filter at the mixer output would isolate the IF.

### 7.2 Intermodulation Products in Real Mixers

Real mixers use nonlinear elements such as diodes, so in addition to the desired sum and difference, **harmonics** and **intermodulation products** appear. On a logarithmic power axis, a "forest" of lines can be seen. These are seen as strong lines corresponding to harmonics of the main signals. Proper signal levels minimize these while preserving the desired mixing products.

In [ ]:
plt.figure(figsize=(10, 4))
plt.semilogy(freqs_dsb/1e3, P_usb + 1e-30)
plt.xlabel("Frequency (kHz)")
plt.ylabel("Power (arb, log)")
plt.title("DSB Mixer Output (Log Scale): Intermodulation and Harmonics")
plt.xlim(-400, 400)
plt.grid(True, alpha=0.3)
plt.show()

### 7.3 The Single-Sideband (SSB) Mixer

An SSB mixer uses two DSB mixers with a 90° phase shift on one LO. The outputs are the **I** (in-phase) and **Q** (quadrature) components. Forming the complex signal $I + iQ$ and computing the power spectrum allows **positive and negative** frequencies to be distinguished. Upper and lower sidebands no longer fold on top of each other.

In [ ]:
ssb_ch1 = np.load("SSB_DATA/ssb_27MHZ_CH1_0.npz")["arr_0"]
ssb_ch2 = np.load("SSB_DATA/ssb_27MHZ_CH2_0.npz")["arr_0"]
I = ssb_ch1[0].astype(float) - np.mean(ssb_ch1[0])
Q = ssb_ch2[0].astype(float) - np.mean(ssb_ch2[0])
x_ssb = I + 1j * Q
N_ssb = len(x_ssb)
fs_ssb = 2.4e6
freqs_ssb = np.fft.fftshift(np.fft.fftfreq(N_ssb, d=1/fs_ssb))
P_ssb = np.fft.fftshift(np.abs(np.fft.fft(x_ssb))**2)

plt.figure(figsize=(10, 4))
plt.semilogy(freqs_ssb/1e6, P_ssb + 1e-30)
plt.xlabel("Frequency (MHz)")
plt.ylabel("Power (arb, log)")
plt.title("SSB Mixer: Power Spectrum of Complex I+jQ (Positive vs Negative Frequency Distinct)")
plt.xlim(-1.5, 1.5)
plt.grid(True, alpha=0.3)
plt.show()

**Figure 18.** Power spectrum of the SSB mixer output (complex $I + jQ$). Positive and negative frequency components are no longer symmetric in the same way as for a real signal. The SSB mixer preserves the distinction between upper and lower sidebands, so we can tell whether $\nu_\mathrm{RF}$ is above or below $\nu_\mathrm{LO}$. This is very important for spectroscopy and avoiding sideband confusion.

---

## 8. On Mixers and the Heterodyne Process (Theory)

Mixers and the **heterodyne** process are at the heart of radio receivers. They shift the incoming spectrum by a fixed LO frequency so that filters and samplers can operate at a convenient, often lower, frequency.

### 8.1 The Heterodyne Process

A **heterodyne** receiver multiplies the RF signal by a **local oscillator** (LO) at $\nu_\mathrm{LO}$. The product contains $\nu_\mathrm{RF} + \nu_\mathrm{LO}$ and $|\nu_\mathrm{RF} - \nu_\mathrm{LO}|$. Low-pass or band-pass filtering selects one of these (usually the **difference**), the **intermediate frequency** (IF). Tuning the receiver means changing $\nu_\mathrm{LO}$. The rest of the chain (filters, ADC) stays fixed. This is how AM radios tune to different stations and how radio telescopes tune to different sky frequencies.

### 8.2 Single-Sideband (SSB) Mixer Theory

With **complex** notation, an ideal SSB mixer uses LO $e^{-i\omega_\mathrm{LO}t}$. An input $E(t) = A\sin(\omega_\mathrm{LO}-\Delta\omega)t + B\sin(\omega_\mathrm{LO}+\Delta\omega)t$ (lower and upper sidebands) becomes, after mixing and low-pass filtering, $A\,e^{-i\Delta\omega t}/(2i) + B\,e^{i\Delta\omega t}/(2i)$. The **real** and **imaginary** parts (I and Q) preserve the distinction: positive and negative $\Delta\nu$ remain separate. So we can tell whether the signal is above or below the LO. Negative frequency in the complex baseband just means the phase of I relative to Q rotates the opposite way.

### 8.3 Double-Sideband (DSB) Mixer Theory

A DSB mixer uses a **real** LO (e.g. $\sin\omega_\mathrm{LO}t$). Then $E(t)\cdot\sin\omega_\mathrm{LO}t$ produces $\cos\Delta\omega t - \cos(2\omega_\mathrm{LO}+\Delta\omega)t$ (and similar for the other sideband). After low-pass filtering we get a **single** beat at $\Delta\nu$. If the input contains **both** upper and lower sidebands, $A\sin(\omega_\mathrm{LO}+\Delta\omega)t + B\sin(\omega_\mathrm{LO}-\Delta\omega)t$, the DSB output is $(A+B)\cos\Delta\omega t$: the two sidebands **add** and cannot be separated. So the DSB mixer is simpler to build but cannot distinguish $\nu_\mathrm{LO}+\Delta\nu$ from $\nu_\mathrm{LO}-\Delta\nu$; the SSB mixer is required when that distinction matters (e.g. spectroscopy, avoiding image confusion).

**AI-based language models were used as a supplementary tool to support conceptual understanding of the experimental methods, to assist in re-learning Python-based data analysis workflows, and to help refine the wording and clarity of data interpretation.